# Сопоставление подписантов JSON ↔ CSV
Для каждого из двух полей JSON выбирается **один** кандидат из CSV. Инициалы должны совпадать; фамилии оцениваются по N-граммам, Левенштейну и Jaro–Winkler. Низкая уверенность отмечается отдельно.

In [ ]:
from pathlib import Path
import csv, json, re, unicodedata
from difflib import SequenceMatcher
import pandas as pd

# === НАСТРОЙКИ ===
JSON_DIR = Path(r"json")
CSV_PATH = Path(r"signers.csv")
JSON_KEYS = ["Подписант1", "Подписант2"]  # укажи два ключа JSON
CSV_FIO_COLUMN = "ФИО"                  # столбец с полным ФИО
CSV_SEPARATOR = ";"                     # например ; или ,
OUTPUT = "matched_signers.xlsx"
MIN_CONFIDENCE = 0.65  # ниже порога результат помечается как требующий проверки


In [ ]:
def norm(s):
    s = unicodedata.normalize("NFKC", str(s or "")).lower().replace("ё", "е")
    return re.sub(r"[^а-яa-z]", "", s)

def parts(s):
    # Фамилия И.О. либо Фамилия Имя Отчество
    tokens = re.findall(r"[А-Яа-яЁёA-Za-z]+", str(s or ""))
    if len(tokens) < 3:
        return None
    return norm(tokens[0]), norm(tokens[1])[0], norm(tokens[2])[0]

def ngram(a, b, size=2):
    def grams(s): return {s[i:i+size] for i in range(max(0, len(s)-size+1))}
    x, y = grams(a), grams(b)
    return 2 * len(x & y) / (len(x) + len(y)) if x or y else 0

def lev_sim(a, b):
    prev = list(range(len(b)+1))
    for i, ca in enumerate(a, 1):
        curr = [i]
        for j, cb in enumerate(b, 1):
            curr.append(min(curr[-1]+1, prev[j]+1, prev[j-1]+(ca != cb)))
        prev = curr
    return 1 - prev[-1] / max(len(a), len(b), 1)

def jaro_winkler(a, b):
    if a == b: return 1.0
    radius = max(0, max(len(a), len(b))//2 - 1)
    am, bm = [False]*len(a), [False]*len(b)
    matches = 0
    for i, ch in enumerate(a):
        for j in range(max(0, i-radius), min(len(b), i+radius+1)):
            if not bm[j] and ch == b[j]:
                am[i] = bm[j] = True
                matches += 1
                break
    if not matches: return 0.0
    aa = [c for c, flag in zip(a, am) if flag]
    bb = [c for c, flag in zip(b, bm) if flag]
    swaps = sum(x != y for x, y in zip(aa, bb))/2
    j = (matches/len(a) + matches/len(b) + (matches-swaps)/matches)/3
    prefix = 0
    for x, y in zip(a[:4], b[:4]):
        if x != y: break
        prefix += 1
    return j + prefix*0.1*(1-j) if j >= 0.7 else j

def score(a, b):
    return 0.30*ngram(a,b) + 0.35*lev_sim(a,b) + 0.35*jaro_winkler(a,b)


In [ ]:
# Читаем справочник с полными ФИО
reference = pd.read_csv(CSV_PATH, sep=CSV_SEPARATOR, encoding="utf-8-sig", dtype=str).fillna("")
if CSV_FIO_COLUMN not in reference.columns:
    raise KeyError(f"Нет столбца {CSV_FIO_COLUMN!r}. Доступны: {list(reference.columns)}")

index = {}
for fio in reference[CSV_FIO_COLUMN]:
    parsed = parts(fio)
    if parsed:
        surname, i1, i2 = parsed
        index.setdefault((i1, i2), []).append((surname, fio))

def match_one(value):
    parsed = parts(value)
    if not parsed:
        return "", None, "Некорректное ФИО"
    surname, i1, i2 = parsed
    candidates = index.get((i1, i2), [])
    if not candidates:
        return "", None, "Нет кандидатов с такими инициалами"
    best_surname, best_fio = max(candidates, key=lambda x: (score(surname, x[0]), x[0] == surname))
    confidence = score(surname, best_surname)
    return best_fio, round(confidence, 4), ("OK" if confidence >= MIN_CONFIDENCE else "Проверить вручную")

rows = []
for path in sorted(JSON_DIR.glob("*.json")):
    with path.open(encoding="utf-8-sig") as f:
        data = json.load(f)
    if not isinstance(data, dict):
        print(f"Пропущен {path.name}: корень JSON не объект")
        continue
    for key in JSON_KEYS:
        original = data.get(key, "")
        found, similarity, status = match_one(original)
        rows.append({"Файл": path.name, "Ключ": key, "Исходное ФИО": original,
                     "Лучший кандидат CSV": found, "Сходство": similarity, "Статус": status})

result = pd.DataFrame(rows)
result.to_excel(OUTPUT, index=False)
print(f"Готово: {OUTPUT}; строк: {len(result)}")
display(result.head(20))
